In [1]:
!nvidia-smi


Mon Sep 14 09:14:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import files
up = files.upload()

Saving mpa_fer.zip to mpa_fer.zip


In [3]:
!unzip -oq mpa_fer.zip -d /content/mpa_fer
%cd /content/mpa_fer
!ls

/content/mpa_fer
configs     losses.py  notebooks  README.md	    tools     utils
dataset.py  models     prompts	  requirements.txt  train.py


In [4]:
!pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00


In [5]:
import clip
print(clip.available_models())

['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']


In [6]:
!python tools/sanity_check.py

torch 2.11.0+cu128 | device: cuda (Tesla T4)

=== 1. Config ===
  [PASS] config loads  (configs/rafdb.yaml)
  [PASS] 7 class names  (['surprise', 'fear', 'disgust', 'happiness', 'sadness', 'anger', 'neutral'])
  [PASS] --set overrides work (int, bool, number keys)
  [PASS] typo in --set key raises an error

=== 2. Hard prompts fit in CLIP's 77 tokens ===
  [INFO] template (type 3): a photo of a person making a facial expression of {cls}, {desc}
  [PASS] every class has a description
  [INFO] surprise   38 tokens
  [INFO] fear       42 tokens
  [INFO] disgust    40 tokens
  [INFO] happiness  40 tokens
  [INFO] sadness    43 tokens
  [INFO] anger      48 tokens
  [INFO] neutral    40 tokens
  [PASS] all hard prompts <= 77 tokens  (longest 48)

=== 3. Build model, CLIP frozen ===
100%|███████████████████████████████████████| 335M/335M [00:03<00:00, 95.0MiB/s]
  [INFO] built in 8.8s (first run downloads CLIP, about 335 MB)
  [INFO] Total params:     124,402,688
  [INFO] Trainable params: 7

In [7]:
import kagglehub
DATA_ROOT = kagglehub.dataset_download('shuvoalok/raf-db-dataset')
print(DATA_ROOT)
!ls -R $DATA_ROOT | head -20

100%|██████████| 37.7M/37.7M [00:00<00:00, 164MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/shuvoalok/raf-db-dataset/versions/2
/root/.cache/kagglehub/datasets/shuvoalok/raf-db-dataset/versions/2:
DATASET
test_labels.csv
train_labels.csv

/root/.cache/kagglehub/datasets/shuvoalok/raf-db-dataset/versions/2/DATASET:
test
train

/root/.cache/kagglehub/datasets/shuvoalok/raf-db-dataset/versions/2/DATASET/test:
1
2
3
4
5
6
7

/root/.cache/kagglehub/datasets/shuvoalok/raf-db-dataset/versions/2/DATASET/test/1:
test_0002_aligned.jpg


In [8]:
from dataset import RAFDB, detect_layout, build_transforms
from utils.misc import load_config
cfg = load_config('configs/rafdb.yaml', [f'data.root={DATA_ROOT}'])
print('layout:', detect_layout(DATA_ROOT))
for split in ['train','test']:
    ds = RAFDB(DATA_ROOT, split, cfg['data']['class_names'],
               transform=build_transforms(cfg, train=False))
    print(split, len(ds), ds.class_counts())

layout: folder
train 12271 [1290, 281, 717, 4772, 1982, 705, 2524]
test 3068 [329, 74, 160, 1185, 478, 162, 680]


In [9]:
!python tools/build_prototypes.py --set data.root=$DATA_ROOT output.dir=outputs/rafdb

Building class prototypes (Eq. 6) with the frozen CLIP encoder...
prototypes: 100% 96/96 [00:37<00:00,  2.57it/s]
  used 12271 images; per class [1290, 281, 717, 4772, 1982, 705, 2524]
  prototype norms: [10.29, 10.77, 10.77, 10.11, 10.42, 10.35, 9.65]
Saved prototypes to outputs/rafdb/prototypes_vitb16_full.pt


In [10]:
print(DATA_ROOT)

/root/.cache/kagglehub/datasets/shuvoalok/raf-db-dataset/versions/2


In [11]:
!python train.py --set data.root=$DATA_ROOT output.dir=outputs/rafdb train.epochs=3

[2026-09-14 09:41:27] experiment: mpa_fer_rafdb_vitb16
[2026-09-14 09:41:27] device: cuda (Tesla T4) | amp: True
[2026-09-14 09:41:27] overrides: ['data.root=/root/.cache/kagglehub/datasets/shuvoalok/raf-db-dataset/versions/2', 'output.dir=outputs/rafdb', 'train.epochs=3']
[2026-09-14 09:41:27] train images: 12271  test images: 3068
[2026-09-14 09:41:27] train class counts: [1290, 281, 717, 4772, 1982, 705, 2524]
[2026-09-14 09:41:31] Total params:     124,402,688
[2026-09-14 09:41:31] Trainable params: 78,848 (0.0634%)
[2026-09-14 09:41:31]   text_prompts.ctx: (10, 512) = 5,120
[2026-09-14 09:41:31]   visual_encoder.prompts: (12, 8, 768) = 73,728
[2026-09-14 09:41:31] trainable size: 0.301 MB in fp32 (paper quotes 0.218 MB for ViT-B/16)
[2026-09-14 09:41:31] Loaded prototypes from outputs/rafdb/prototypes_vitb16_full.pt
[2026-09-14 09:41:31] training for 3 epochs, 383 iters/epoch
epoch 1:   0% 0/383 [00:00<?, ?it/s]/content/mpa_fer/train.py:123: UserWarning: Detected call of `lr_sched

In [12]:
!python train.py --set data.root=$DATA_ROOT output.dir=outputs/clip_off \
    train.epochs=3 train.grad_clip=null output.resume=false

[2026-09-14 09:52:12] experiment: mpa_fer_rafdb_vitb16
[2026-09-14 09:52:12] device: cuda (Tesla T4) | amp: True
[2026-09-14 09:52:12] overrides: ['data.root=/root/.cache/kagglehub/datasets/shuvoalok/raf-db-dataset/versions/2', 'output.dir=outputs/clip_off', 'train.epochs=3', 'train.grad_clip=null', 'output.resume=false']
[2026-09-14 09:52:12] train images: 12271  test images: 3068
[2026-09-14 09:52:12] train class counts: [1290, 281, 717, 4772, 1982, 705, 2524]
[2026-09-14 09:52:17] Total params:     124,402,688
[2026-09-14 09:52:17] Trainable params: 78,848 (0.0634%)
[2026-09-14 09:52:17]   text_prompts.ctx: (10, 512) = 5,120
[2026-09-14 09:52:17]   visual_encoder.prompts: (12, 8, 768) = 73,728
[2026-09-14 09:52:17] trainable size: 0.301 MB in fp32 (paper quotes 0.218 MB for ViT-B/16)
[2026-09-14 09:52:17] Building class prototypes (Eq. 6) with the frozen CLIP encoder...
prototypes: 100% 96/96 [00:36<00:00,  2.61it/s]
  used 12271 images; per class [1290, 281, 717, 4772, 1982, 705, 2